# PaliGemma 2 ile Türkçe sahne metni OCR deneyi

**Soru:** PaliGemma 2 (3B, mix) Türkçe sahne metnini ne kadar doğru okuyor? 224 ve 448 çözünürlüğü arasında fark var mı? Prompt dili sonucu değiştiriyor mu?

**Önce yap (bir kere):**
1. Hugging Face'te iki model sayfasında lisansı kabul et: `google/paligemma2-3b-mix-224` ve `google/paligemma2-3b-mix-448` (sayfalar kapılı, kabul etmezsen indirme hata verir).
2. Hugging Face'ten bir *read* token oluştur, Colab'da sol menüdeki 🔑 **Secrets** bölümüne `HF_TOKEN` adıyla ekle ve "Notebook access"i aç.
3. Çalışma zamanı: **T4 GPU** (Runtime > Change runtime type).
4. `veriseti.zip` dosyasını Colab'ın dosya panelinden yükle.

In [ ]:
!pip install -q "transformers>=4.50" accelerate pandas

In [ ]:
import os, re, time, unicodedata, zipfile, gc
import torch, pandas as pd
from PIL import Image
from google.colab import userdata
from huggingface_hub import login
from transformers import PaliGemmaProcessor, PaliGemmaForConditionalGeneration

login(token=userdata.get("HF_TOKEN"))

if not os.path.exists("veriseti"):
    zipfile.ZipFile("veriseti.zip").extractall(".")
print("GPU:", torch.cuda.get_device_name(0), "| bellek:", round(torch.cuda.get_device_properties(0).total_memory/1e9,1), "GB")

## Ayarlar

- `IMAGE_SOURCE = "orijinal"`: görseller olduğu gibi verilir, işlemci (processor) her modelin kendi çözünürlüğüne (224 ya da 448) küçültür. Bu, modelin normal çalışma şeklidir.
- `IMAGE_SOURCE = "letterbox"`: hazır 224/448 klasörleri kullanılır (en-boy oranı korunmuş, kenarlar dolgulu). İkisini de çalıştırıp karşılaştırmak ayrı bir bulgu olabilir.
- **Kullandığın yöntemi yazıda mutlaka belirt.**
- Veri tipi: T4 `bfloat16`'yı yerel desteklemez. Önce `float16` denenir; çıktılar boş, bozuk ya da `nan` gelirse `DTYPE = torch.float32` yap (3B model ~12 GB tutar, T4'e zor sığar ama sığabilir).

In [ ]:
IMAGE_SOURCE = "orijinal"          # "orijinal" ya da "letterbox"
GROUPS = ["ana"]                   # ["ana", "taninmis_marka", "kontrol"] ekleyebilirsin
ONLY_VERIFIED = False              # True: sadece manifest'te dogrulandi=x olan satirlar
MAX_NEW_TOKENS = 256

MODELS = {
    224: "google/paligemma2-3b-mix-224",
    448: "google/paligemma2-3b-mix-448",
}
PROMPTS = {
    "ocr":    "ocr",
    "tr_soru": "answer tr Görselde yazan tüm metni yaz.",
    "en_soru": "answer en What is all the text written in the image?",
}
DTYPE = torch.float16 if torch.cuda.get_device_capability(0)[0] < 8 else torch.bfloat16
print("dtype:", DTYPE)

manifest = pd.read_csv("veriseti/manifest.csv", encoding="utf-8-sig").fillna("")
manifest = manifest[manifest["grup"].isin(GROUPS)]
if ONLY_VERIFIED:
    manifest = manifest[manifest["dogrulandi"].str.lower() == "x"]
print(len(manifest), "gorsel")

def image_path(row, res):
    folder = "orijinal" if IMAGE_SOURCE == "orijinal" else str(res)
    return f"veriseti/{row['grup']}/{folder}/{row['dosya']}"

## Modelleri çalıştır

Her model sırayla yüklenir, tüm görseller ve promptlar çalıştırılır, sonra bellek boşaltılır. Sonuçlar `sonuclar.csv` dosyasına kaydedilir. Yaklaşık 10-20 dakika sürebilir. Yarıda kalırsa `sonuclar.csv` dosyasında o ana kadarki satırlar durur.

In [ ]:
results = []
for res, model_id in MODELS.items():
    print(f"\n=== {model_id} ===")
    processor = PaliGemmaProcessor.from_pretrained(model_id)
    model = PaliGemmaForConditionalGeneration.from_pretrained(
        model_id, torch_dtype=DTYPE, device_map="auto").eval()
    for _, row in manifest.iterrows():
        image = Image.open(image_path(row, res)).convert("RGB")
        for pname, prompt in PROMPTS.items():
            inputs = processor(text=prompt, images=image, return_tensors="pt").to(DTYPE).to(model.device)
            n_in = inputs["input_ids"].shape[-1]
            t0 = time.time()
            with torch.inference_mode():
                out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
            text = processor.decode(out[0][n_in:], skip_special_tokens=True)
            results.append(dict(cozunurluk=res, prompt=pname, dosya=row["dosya"], grup=row["grup"],
                                cikti=text, sure_sn=round(time.time()-t0, 1)))
        print(row["dosya"], "tamam")
        pd.DataFrame(results).to_csv("sonuclar.csv", index=False, encoding="utf-8-sig")
    del model, processor
    gc.collect(); torch.cuda.empty_cache()

res_df = pd.DataFrame(results)
res_df.to_csv("sonuclar.csv", index=False, encoding="utf-8-sig")
res_df.head(6)

## Hızlı bakış

Ölçüme geçmeden önce birkaç çıktıya kendi gözünle bak. Boş, bozuk ya da tekrar eden çıktı varsa (özellikle float16'da) `DTYPE`'ı değiştirip yeniden çalıştır.

In [ ]:
pd.set_option("display.max_colwidth", 200)
sample = res_df[res_df["dosya"].isin(manifest["dosya"].head(3))]
sample[["cozunurluk","prompt","dosya","cikti"]]

## Ölçüm

Ground truth'taki her satır (`|` ile ayrılmış parçalar) için model çıktısında o metnin bulunup bulunmadığına bakılır. Çıktı genellikle görseldeki *tüm* metni içerdiği için tüm çıktıyla karakter hatası (CER) hesaplamak yanıltıcı olur; bunun yerine **parça bazlı** ölçüm yapılır:

- **tam_bulundu:** Parça çıktıda aynen (küçük/büyük harf ve noktalama farkı yok sayılarak) geçiyor mu? `İ/I` ve `ı/i` farkı **korunur**.
- **aksansiz_bulundu:** Aynı kontrol, ama ç/ğ/ı/ö/ş/ü → c/g/i/o/s/u çevrilerek. İkisi arasındaki fark, "model harfi okuyor ama Türkçe işaretini kaçırıyor" demektir.
- **hata_orani:** Parçanın çıktıdaki en yakın konumla arasındaki karakter düzeyinde hata (0 = birebir).

In [ ]:
TR_FOLD = str.maketrans("çğıöşüâî", "cgiosuai")

def tr_lower(s):
    return s.replace("İ", "i").replace("I", "ı").lower()

def norm(s):
    s = unicodedata.normalize("NFC", str(s))
    s = tr_lower(s)
    s = re.sub(r"[^\w]+", " ", s, flags=re.UNICODE).replace("_", " ")
    return re.sub(r"\s+", " ", s).strip()

def fold(s):
    return s.translate(TR_FOLD)

def approx_substring_dist(ref, text):
    # ref'in text icindeki en iyi (en az hatali) konumla duzenleme mesafesi
    m, n = len(ref), len(text)
    if m == 0: return 0
    prev = [0] * (n + 1)
    for i in range(1, m + 1):
        cur = [i] + [0] * n
        for j in range(1, n + 1):
            cost = 0 if ref[i-1] == text[j-1] else 1
            cur[j] = min(prev[j] + 1, cur[j-1] + 1, prev[j-1] + cost)
        prev = cur
    return min(prev)

def score(res_df, manifest):
    gt = manifest.set_index("dosya")
    res_df = res_df[res_df["dosya"].isin(gt.index)]   # manifest'ten cikarilan gorselleri atla
    rows = []
    for _, r in res_df.iterrows():
        out_n = norm(r["cikti"]); out_f = fold(out_n)
        for kind in ("ana", "ikincil"):
            for seg in [s.strip() for s in str(gt.loc[r["dosya"], kind + "_metin"]).split("|") if s.strip()]:
                ref = norm(seg)
                if not ref: continue
                rows.append(dict(
                    cozunurluk=r["cozunurluk"], prompt=r["prompt"], dosya=r["dosya"], tur=kind, parca=seg,
                    tam_bulundu=ref in out_n,
                    aksansiz_bulundu=fold(ref) in out_f,
                    hata_orani=round(approx_substring_dist(ref, out_n) / len(ref), 3)))
    return pd.DataFrame(rows)

sk = score(res_df, manifest)
sk.to_csv("skorlar_parca.csv", index=False, encoding="utf-8-sig")
sk.head()

In [ ]:
ozet = (sk.groupby(["tur", "cozunurluk", "prompt"])
          .agg(parca_sayisi=("parca", "count"),
               tam_bulundu=("tam_bulundu", "mean"),
               aksansiz_bulundu=("aksansiz_bulundu", "mean"),
               ort_hata_orani=("hata_orani", "mean"))
          .round(3).reset_index())
ozet.to_csv("ozet.csv", index=False, encoding="utf-8-sig")
ozet

## Görsel bazında karşılaştırma (224 ve 448)

Her görsel için `ocr` promptunda ana metnin kaç parçasının bulunduğunu gösterir. Farkın çıktığı görseller yazıda anlatmaya değer örneklerdir.

In [ ]:
ana = sk[(sk["tur"] == "ana") & (sk["prompt"] == "ocr")]
gorsel = ana.pivot_table(index="dosya", columns="cozunurluk", values="tam_bulundu", aggfunc="mean").round(2)
gorsel["fark_448_224"] = gorsel[448] - gorsel[224]
gorsel.sort_values("fark_448_224", ascending=False)

## Türkçe harf hataları

Model harfi okumuş ama aksanı kaçırmışsa (`aksansiz_bulundu` doğru, `tam_bulundu` yanlış) burada listelenir. Çıktıyı görselle karşılaştırıp hata türünü (İ→I, ş→s, ğ→g...) elle not et.

In [ ]:
aksan = sk[(sk["aksansiz_bulundu"]) & (~sk["tam_bulundu"])]
print(len(aksan), "parcada aksan/harf hatasi")
aksan[["cozunurluk","prompt","dosya","parca"]].sort_values("dosya")

## Yazıya taşınacaklar

- `ozet.csv`: 224/448 ve prompt karşılaştırması
- `skorlar_parca.csv`: parça bazlı ham sonuçlar
- `sonuclar.csv`: modelin ham çıktıları (yazıya 2-3 örnek koy, görselle yan yana)

**Dürüst raporlama için:** kaç görsel, kaç parça ölçtüğünü, ground truth'u kimin çıkardığını, `IMAGE_SOURCE` ve `DTYPE` ayarını ve modelin `mix` sürümü olduğunu yaz.

In [ ]:
from google.colab import files
for f in ["sonuclar.csv", "skorlar_parca.csv", "ozet.csv"]:
    files.download(f)